# Actividad 3 - SVM y K-Means
**Dataset:** Breast Cancer Wisconsin  
**Fuente:** scikit-learn  
**Fecha de consulta:** 12 de septiembre de 2026


In [ ]:
# 1. Importar librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    silhouette_score
)


In [ ]:
# 2. Cargar el dataset Breast Cancer Wisconsin

cancer = load_breast_cancer()

X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name="target")

df = X.copy()
df["target"] = y

df.head()


In [ ]:
# 3. Información general del dataset

print("Número de registros:", df.shape[0])
print("Número de columnas:", df.shape[1])

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores faltantes:")
print(df.isnull().sum())

print("\nDuplicados:")
print(df.duplicated().sum())

print("\nDistribución de la variable objetivo:")
print(df["target"].value_counts())

print("\n0 = maligno")
print("1 = benigno")


In [ ]:
# 4. Separar variables predictoras y variable objetivo

X = df.drop(columns="target")
y = df["target"]

# Copia de las etiquetas reales para comparación posterior con K-Means
y_real = y.copy()

print("X:", X.shape)
print("y:", y.shape)


In [ ]:
# 5. División de entrenamiento y prueba
# 80% entrenamiento y 20% prueba
# Se utiliza stratify para conservar la proporción de clases.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


In [ ]:
# 6. SVM con kernel lineal
# StandardScaler y SVM se integran dentro de un Pipeline.

svm_lineal = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear"))
])

svm_lineal.fit(X_train, y_train)

y_pred_lineal = svm_lineal.predict(X_test)

acc_lineal = accuracy_score(y_test, y_pred_lineal)
prec_lineal = precision_score(y_test, y_pred_lineal)
rec_lineal = recall_score(y_test, y_pred_lineal)
f1_lineal = f1_score(y_test, y_pred_lineal)

print("Accuracy:", acc_lineal)
print("Precision:", prec_lineal)
print("Recall:", rec_lineal)
print("F1-score:", f1_lineal)


In [ ]:
# 7. SVM con kernel RBF

svm_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

svm_rbf.fit(X_train, y_train)

y_pred_rbf = svm_rbf.predict(X_test)

acc_rbf = accuracy_score(y_test, y_pred_rbf)
prec_rbf = precision_score(y_test, y_pred_rbf)
rec_rbf = recall_score(y_test, y_pred_rbf)
f1_rbf = f1_score(y_test, y_pred_rbf)

print("Accuracy:", acc_rbf)
print("Precision:", prec_rbf)
print("Recall:", rec_rbf)
print("F1-score:", f1_rbf)


In [ ]:
# 8. Tabla comparativa de modelos SVM

resultados_svm = pd.DataFrame({
    "Modelo SVM": ["Modelo 1", "Modelo 2"],
    "Kernel": ["Lineal", "RBF"],
    "Accuracy": [acc_lineal, acc_rbf],
    "Precision": [prec_lineal, prec_rbf],
    "Recall": [rec_lineal, rec_rbf],
    "F1-score": [f1_lineal, f1_rbf]
})

resultados_svm.round(4)


In [ ]:
# 9. Matriz de confusión del modelo con mejor F1-score

if f1_rbf >= f1_lineal:
    mejor_pred = y_pred_rbf
    mejor_nombre = "SVM RBF"
else:
    mejor_pred = y_pred_lineal
    mejor_nombre = "SVM Lineal"

cm = confusion_matrix(y_test, mejor_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Maligno", "Benigno"]
)

disp.plot()
plt.title(f"Matriz de confusión - {mejor_nombre}")
plt.show()

print(cm)


In [ ]:
# 10. Modificación del hiperparámetro C en SVM RBF

valores_C = [0.1, 1, 10]
resultados_C = []

for C in valores_C:
    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=C))
    ])

    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    resultados_C.append({
        "C": C,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-score": f1_score(y_test, pred)
    })

pd.DataFrame(resultados_C).round(4)


In [ ]:
# 11. Escalamiento para K-Means
# La variable objetivo NO se usa para entrenar K-Means.

scaler_kmeans = StandardScaler()
X_scaled = scaler_kmeans.fit_transform(X)

print(X_scaled.shape)


In [ ]:
# 12. Evaluar diferentes valores de k con método del codo y Silhouette Score

inercias = []
silhouettes = []

valores_k = range(2, 7)

for k in valores_k:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    clusters_temp = kmeans.fit_predict(X_scaled)

    inercias.append(kmeans.inertia_)
    silhouettes.append(
        silhouette_score(X_scaled, clusters_temp)
    )

evaluacion_k = pd.DataFrame({
    "k": list(valores_k),
    "Inercia": inercias,
    "Silhouette Score": silhouettes
})

evaluacion_k.round(4)


In [ ]:
# 13. Gráfica del método del codo

plt.figure(figsize=(7, 5))
plt.plot(list(valores_k), inercias, marker="o")

plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")

plt.show()


In [ ]:
# 14. Gráfica de Silhouette Score

plt.figure(figsize=(7, 5))
plt.plot(list(valores_k), silhouettes, marker="o")

plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")

plt.show()


In [ ]:
# 15. Modelo K-Means final con k = 2

kmeans_final = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

clusters = kmeans_final.fit_predict(X_scaled)

df_clusters = X.copy()
df_clusters["Cluster"] = clusters

df_clusters.head()


In [ ]:
# 16. Tamaño de cada cluster

tamano_clusters = (
    df_clusters["Cluster"]
    .value_counts()
    .sort_index()
)

tamano_clusters


In [ ]:
# 17. Promedio de variables principales por cluster

variables_principales = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean concavity",
    "mean concave points"
]

resumen_clusters = (
    df_clusters.groupby("Cluster")[variables_principales]
    .mean()
    .round(3)
)

resumen_clusters["Tamaño"] = tamano_clusters

resumen_clusters


In [ ]:
# 18. PCA para reducir los datos a dos dimensiones

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Cluster": clusters
})

print("Varianza explicada:")
print(pca.explained_variance_ratio_)

print("\nVarianza acumulada:")
print(pca.explained_variance_ratio_.sum())


In [ ]:
# 19. Visualización de clusters con PCA

plt.figure(figsize=(8, 6))

for cluster in np.unique(clusters):
    plt.scatter(
        pca_df.loc[pca_df["Cluster"] == cluster, "PC1"],
        pca_df.loc[pca_df["Cluster"] == cluster, "PC2"],
        label=f"Cluster {cluster}",
        alpha=0.7
    )

plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Clusters de K-Means proyectados con PCA")
plt.legend()

plt.show()


In [ ]:
# 20. Visualización de clases reales con PCA
# Las etiquetas reales solo se usan aquí para comparación visual.

pca_real = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Clase real": y_real.values
})

plt.figure(figsize=(8, 6))

for clase in np.unique(y_real):
    plt.scatter(
        pca_real.loc[pca_real["Clase real"] == clase, "PC1"],
        pca_real.loc[pca_real["Clase real"] == clase, "PC2"],
        label=cancer.target_names[clase],
        alpha=0.7
    )

plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Clases reales proyectadas con PCA")
plt.legend()

plt.show()


In [ ]:
# 21. Comparación de clusters con las clases reales

comparacion = pd.crosstab(
    clusters,
    y_real,
    rownames=["Cluster"],
    colnames=["Clase real"]
)

comparacion.columns = ["Maligno", "Benigno"]

comparacion


In [ ]:
# 22. Resultados finales resumidos

print("MEJOR MODELO SVM:", mejor_nombre)
print()

print("Resultados SVM:")
display(resultados_svm.round(4))

print("\nEvaluación de K-Means:")
display(evaluacion_k.round(4))

print("\nComparación de clusters con clases reales:")
display(comparacion)


## Interpretación final

El modelo SVM permite predecir una clase conocida, benigno o maligno, utilizando las características numéricas de los tumores. El kernel RBF suele presentar el mejor desempeño en este conjunto de datos.

K-Means trabaja sin utilizar la variable objetivo y permite identificar grupos naturales según la similitud entre las observaciones. Los clusters encontrados presentan una correspondencia parcial con las clases reales, pero no tienen por qué coincidir exactamente porque K-Means agrupa por distancia y no por etiquetas conocidas.

El aprendizaje supervisado es adecuado cuando se desea predecir una categoría conocida. El aprendizaje no supervisado es útil cuando se busca explorar patrones o estructuras sin etiquetas previas.
